# Plant disease ML: Logistic Regression on real leaf images
Runs top to bottom in Google Colab (**Runtime → Run all**). Takes about 2–3 minutes, mostly the download.

1. Download 5 tomato classes from PlantVillage (about 100 MB)
2. Turn every leaf photo into 53 numbers (colour, texture, shape)
3. Compare 6 classical ML models, as in Paper 1 (Kiran & Chandrappa, 2023)
4. Draw all 9 logistic regression graphs, with a short note under each

Every number matches the presentation because the random seeds are fixed.

## 1 · Get the images
Only the 5 tomato folders are downloaded, not the whole 1 GB repository. The cell skips the download if the images are already there.

In [ ]:
import os, glob
BASE = '/content' if os.path.isdir('/content') else os.getcwd()
if not glob.glob(f'{BASE}/**/Tomato___healthy', recursive=True):
    !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/spMohanty/PlantVillage-Dataset.git {BASE}/pv
    !cd {BASE}/pv && git sparse-checkout set --no-cone "/raw/color/Tomato___healthy/*" "/raw/color/Tomato___Early_blight/*" "/raw/color/Tomato___Late_blight/*" "/raw/color/Tomato___Septoria_leaf_spot/*" "/raw/color/Tomato___Tomato_Yellow_Leaf_Curl_Virus/*" > /dev/null 2>&1
os.chdir(BASE)

# find the folder that holds the class folders (works for PlantVillage or the Kaggle train/ folder)
hits = [h for h in glob.glob(f'{BASE}/**/Tomato___healthy', recursive=True) if 'valid' not in h.lower()]
ROOT = os.path.dirname(sorted(hits, key=len)[0])
print('Using images from:', ROOT)
for d in ['Tomato___healthy','Tomato___Early_blight','Tomato___Late_blight','Tomato___Septoria_leaf_spot','Tomato___Tomato_Yellow_Leaf_Curl_Virus']:
    print(f'  {d:42s} {len(glob.glob(f"{ROOT}/{d}/*")):5d} images')
os.makedirs('fig', exist_ok=True)

## 2 · Turn each leaf into 53 numbers
For each photo: resize to 256×256, blur lightly to remove noise, and cut out the leaf with a green + brown HSV mask (Paper 1's segmentation idea). Then measure:

| Group | Count | What it captures |
|---|---|---|
| RGB mean and std | 6 | average colour and its spread (Unit 1: mean, variance) |
| GLI, G/R | 2 | greenness indices |
| HSV histogram | 32 | share of the leaf in each colour range (Unit 1: densities) |
| Haralick (GLCM) texture | 6 | smooth vs rough vs spotty |
| Hu moments | 7 | shape of the outline |

It also records the brown-area fraction, which graph 1 uses.

In [ ]:
import numpy as np, cv2, glob, os, random, json
from skimage.feature import graycomatrix, graycoprops
random.seed(7); np.random.seed(7)
CLASSES={'Tomato___healthy':'Healthy','Tomato___Early_blight':'Early blight','Tomato___Late_blight':'Late blight',
 'Tomato___Septoria_leaf_spot':'Septoria leaf spot','Tomato___Tomato_Yellow_Leaf_Curl_Virus':'Yellow leaf curl virus'}
N=300
def leaf_mask(img):
    # Paper 1 style: green + brown threshold masks in HSV, union
    hsv=cv2.cvtColor(img,cv2.COLOR_BGR2HSV)
    green=cv2.inRange(hsv,(25,40,40),(95,255,255))
    brown=cv2.inRange(hsv,(5,40,20),(25,255,255))
    m=cv2.bitwise_or(green,brown)
    m=cv2.morphologyEx(m,cv2.MORPH_CLOSE,np.ones((7,7),np.uint8))
    return m,green,brown
def feats(path):
    img=cv2.imread(path); img=cv2.resize(img,(256,256))
    img=cv2.GaussianBlur(img,(3,3),0)
    m,green,brown=leaf_mask(img)
    px=img[m>0].astype(float)  # BGR
    B,G,R=px[:,0],px[:,1],px[:,2]
    rgb=[R.mean(),G.mean(),B.mean(),R.std(),G.std(),B.std()]
    gli=((2*G-R-B)/(2*G+R+B+1e-6)).mean()
    gr=(G/(R+1e-6)).mean()
    hsv=cv2.cvtColor(img,cv2.COLOR_BGR2HSV)
    hist=cv2.calcHist([hsv],[0,1,2],m,[8,2,2],[0,180,0,256,0,256]).flatten(); hist/=hist.sum()+1e-9
    gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY); gray[m==0]=0
    g=graycomatrix((gray//8).astype(np.uint8),[1,2],[0,np.pi/4,np.pi/2,3*np.pi/4],levels=32,symmetric=True,normed=True)
    har=[graycoprops(g,p).mean() for p in ['contrast','dissimilarity','homogeneity','energy','correlation','ASM']]
    hu=cv2.HuMoments(cv2.moments(m)).flatten(); hu=-np.sign(hu)*np.log10(np.abs(hu)+1e-30)
    lesion=(brown>0).sum()/max((m>0).sum(),1)
    return rgb+[gli,gr]+list(hist)+har+list(hu), lesion
X=[];y=[];L=[];paths=[]
for k,(d,name) in enumerate(CLASSES.items()):
    fs=sorted(glob.glob(f'{ROOT}/{d}/*')); fs=random.sample(fs,min(N,len(fs)))
    for f in fs:
        v,les=feats(f); X.append(v); y.append(k); L.append(les); paths.append(f)
np.savez('data.npz',X=np.array(X),y=np.array(y),L=np.array(L),paths=np.array(paths),names=np.array(list(CLASSES.values())))
print('feature matrix:', np.array(X).shape, '(leaves x features)')


## 3 · Compare six classical models (Paper 1's line-up)
Each model gets the same 53 features, standardised, and is scored with 5-fold stratified cross-validation. The second block is the feature ablation from slide 14.

In [ ]:
import numpy as np, json
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix
d=np.load('data.npz'); X,y=d['X'],d['y']
cv=StratifiedKFold(5,shuffle=True,random_state=0)
models={'Logistic Regression':LogisticRegression(max_iter=5000,C=1.0),
 'LDA':LinearDiscriminantAnalysis(),
 'SVM (linear kernel)':SVC(kernel='linear',C=1),
 'SVM (RBF kernel)':SVC(kernel='rbf',C=10,gamma='scale'),
 'KNN (k=5)':KNeighborsClassifier(5),
 'Random Forest':RandomForestClassifier(300,random_state=0)}
res={}
for n,m in models.items():
    p=make_pipeline(StandardScaler(),m); s=cross_val_score(p,X,y,cv=cv)
    res[n]=(s.mean()*100,s.std()*100); print(f'{n:22s} {s.mean()*100:.2f} ± {s.std()*100:.2f}')
# colour-only vs colour+texture for SVM RBF
colour=list(range(0,8))+list(range(8,40)); tex=list(range(40,46)); hu=list(range(46,53))
for lab,idx in [('RGB stats only',list(range(0,6))),('Colour (RGB+indices+HSV hist)',colour),('Colour + texture',colour+tex),('Colour + texture + shape',colour+tex+hu)]:
    s=cross_val_score(make_pipeline(StandardScaler(),SVC(kernel='rbf',C=10)),X[:,idx],y,cv=cv); res['abl_'+lab]=(s.mean()*100,s.std()*100); print(lab,round(s.mean()*100,2))
pred=cross_val_predict(make_pipeline(StandardScaler(),SVC(kernel='rbf',C=10)),X,y,cv=cv)
cm=confusion_matrix(y,pred); np.save('cm.npy',cm); print(cm)
json.dump(res,open('res.json','w'),indent=1)
# Unit1: correlation of mean R,G,B
M=X[:,0:3]; C=np.corrcoef(M.T); print('corr',C.round(3)); print('cov',np.cov(M.T).round(1))
# VIF
vif=[]
for j in range(3):
    o=[k for k in range(3) if k!=j]; r2=LinearRegression().fit(M[:,o],M[:,j]).score(M[:,o],M[:,j]); vif.append(1/(1-r2))
print('VIF',np.round(vif,2)); json.dump({'corr':C.tolist(),'vif':vif},open('u1.json','w'))


## 4 · Logistic Regression: compute everything
This cell fits every logistic regression variant the graphs need and prints the full classification report.

In [ ]:

import numpy as np, json, glob, warnings
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from matplotlib.colors import ListedColormap
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score, learning_curve, validation_curve
from sklearn.metrics import (confusion_matrix, roc_curve, auc, precision_recall_fscore_support,
                             accuracy_score, log_loss, classification_report)
warnings.filterwarnings('ignore')
INK, MUTE, GRID = '#1F2A22', '#6B7266', '#E6E9E4'
PAL = ['#2A9160', '#C9760E', '#3867C9', '#CC3F7A', '#8A6A12']   # validated categorical palette
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10, 'axes.edgecolor': '#BDB6A4',
                     'axes.labelcolor': INK, 'xtick.color': MUTE, 'ytick.color': MUTE, 'text.color': INK,
                     'axes.spines.top': False, 'axes.spines.right': False, 'axes.titlelocation': 'left',
                     'axes.titlesize': 11.5, 'axes.titleweight': 'bold'})

os.makedirs('fig', exist_ok=True)
d = np.load('data.npz', allow_pickle=True)
X, y, L, names = d['X'], d['y'], d['L'], list(d['names'])
short = ['Healthy', 'Early blight', 'Late blight', 'Septoria', 'YLCV']
feat = ['R mean', 'G mean', 'B mean', 'R std', 'G std', 'B std', 'GLI', 'G/R'] + \
       [f'HSV h{h} s{s} v{v}' for h in range(8) for s in range(2) for v in range(2)] + \
       ['Contrast', 'Dissimilarity', 'Homogeneity', 'Energy', 'Correlation', 'ASM'] + [f'Hu {i}' for i in range(1, 8)]
cv = StratifiedKFold(5, shuffle=True, random_state=0)
LR = lambda C=1.0: make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
out = {}

# ---------------- multi-class, all 53 features ----------------
proba = cross_val_predict(LR(), X, y, cv=cv, method='predict_proba')
pred = proba.argmax(1)
out['acc_5class'] = accuracy_score(y, pred); out['logloss_5class'] = log_loss(y, proba)
print('=== 5-class Logistic Regression, 53 features, 5-fold CV ===')
print(f'accuracy {out["acc_5class"]*100:.2f}%   log-loss {out["logloss_5class"]:.3f}')
print(classification_report(y, pred, target_names=short, digits=3))

# ---------------- binary: healthy vs diseased ----------------
yb = (y != 0).astype(int)                                # 1 = diseased
xb = np.log10(L * 100 + 0.5).reshape(-1, 1)              # one feature: brown area (log %)
lr1 = LogisticRegression(class_weight='balanced').fit(xb, yb)   # balanced = equal priors, else it just predicts the 80% majority
w, b0 = lr1.coef_[0, 0], lr1.intercept_[0]
x50 = -b0 / w
out['binary1_w'] = w; out['binary1_b'] = b0; out['binary1_boundary_pct'] = 10 ** x50 - 0.5
pb_all = cross_val_predict(LR(), X, yb, cv=cv, method='predict_proba')[:, 1]
out['acc_binary_53'] = accuracy_score(yb, pb_all > .5)
from sklearn.metrics import balanced_accuracy_score
pb_1 = cross_val_predict(LogisticRegression(class_weight='balanced'), xb, yb, cv=cv, method='predict_proba')[:, 1]
out['acc_binary_1'] = balanced_accuracy_score(yb, pb_1 > .5)
print(f'=== Binary healthy vs diseased ===\n1 feature (brown %): w={w:.3f}, b={b0:.3f}, P=0.5 at {out["binary1_boundary_pct"]:.2f}% brown, CV balanced acc {out["acc_binary_1"]*100:.2f}%')
print(f'all 53 features: CV acc {out["acc_binary_53"]*100:.2f}%  (balanced {balanced_accuracy_score(yb, pb_all > .5)*100:.2f}%)')



In [ ]:
fig_list = []
def save(fig, name): fig.savefig(f'fig/{name}.png', dpi=180, facecolor='white'); fig_list.append(name); plt.close(fig)

def p_sigmoid(ax):
    rng = np.random.default_rng(0)
    jit = rng.uniform(-0.04, 0.04, len(yb))
    ax.scatter(xb[yb == 0, 0], yb[yb == 0] + jit[yb == 0], s=7, color=PAL[0], alpha=.45, lw=0, label='Healthy (y = 0)')
    ax.scatter(xb[yb == 1, 0], yb[yb == 1] + jit[yb == 1], s=7, color=PAL[1], alpha=.35, lw=0, label='Diseased (y = 1)')
    xs = np.linspace(xb.min() - .1, xb.max() + .1, 400)
    ax.plot(xs, 1 / (1 + np.exp(-(w * xs + b0))), color=INK, lw=2.2)
    ax.axhline(.5, color=GRID, lw=1); ax.axvline(x50, color=MUTE, ls=':', lw=1)
    ax.text(x50 + .06, .40, f'P = 0.5 at\n{10**x50-0.5:.1f}% brown', fontsize=8.5, color=MUTE)
    ax.text(xs[0] + .02, .80, f'P(diseased) = 1 / (1 + e^-({w:.2f}·x {b0:+.2f}))', fontsize=8.5)
    tk = [0, 1, 2, 5, 10, 25, 100]; ax.set_xticks([np.log10(t + .5) for t in tk]); ax.set_xticklabels(tk)
    ax.set_xlabel('x = brown area, % of leaf (log scale)'); ax.set_ylabel('P(diseased)')
    ax.set_title('1 · Sigmoid curve (healthy vs diseased, one feature)')
    ax.text(xs[-1], .2, f'balanced accuracy {out["acc_binary_1"]*100:.0f}% with 1 feature\nvs {out["acc_binary_53"]*100:.0f}% with all 53', fontsize=8, color=MUTE, ha='right'); ax.legend(frameon=False, fontsize=8, loc='lower right', markerscale=2)

# 2-D view: two real features (mean Blue, GLI) - the best pair we found, still only ~50%
from sklearn.model_selection import cross_val_score
sc2 = StandardScaler().fit(X[:, [2, 6]]); Z = sc2.transform(X[:, [2, 6]])
lr2 = LogisticRegression(max_iter=5000).fit(Z, y)
out['acc_2feat_cv'] = cross_val_score(LR(), X[:, [2, 6]], y, cv=cv).mean()
def p_regions(ax):
    gx, gy = np.meshgrid(np.linspace(Z[:, 0].min() - 1, Z[:, 0].max() + 1, 400), np.linspace(Z[:, 1].min() - 1, Z[:, 1].max() + 1, 400))
    G = lr2.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    ax.contourf(gx, gy, G, levels=np.arange(-.5, 5), colors=[c + '26' for c in PAL])
    ax.contour(gx, gy, G, levels=np.arange(.5, 4), colors=INK, linewidths=.7)
    for k in range(5): ax.scatter(Z[y == k, 0], Z[y == k, 1], s=6, color=PAL[k], lw=0, alpha=.75, label=short[k])
    ax.set_xlabel('mean Blue (standardised)'); ax.set_ylabel('GLI greenness (standardised)')
    ax.set_title(f'2 · Decision regions, 2 features only ({out["acc_2feat_cv"]*100:.0f}% CV)'); ax.legend(frameon=False, fontsize=7.5, markerscale=2.5, loc='upper right')
    ax.text(.02, .02, 'straight boundaries = linear classifier.\nTwo features overlap too much; 53 get 80%', transform=ax.transAxes, fontsize=8, color=MUTE)

def p_cm(ax):
    cm = confusion_matrix(y, pred); cmn = cm / cm.sum(1, keepdims=True)
    ax.imshow(cmn, cmap=ListedColormap(plt.cm.Greens(np.linspace(.03, .95, 256))), vmin=0, vmax=1)
    for i in range(5):
        for j in range(5): ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=9.5, color='white' if cmn[i, j] > .55 else INK)
    lab = ['Healthy', 'Early\nblight', 'Late\nblight', 'Septoria', 'YLCV']
    ax.set_xticks(range(5)); ax.set_xticklabels(lab, fontsize=8); ax.set_yticks(range(5)); ax.set_yticklabels(lab, fontsize=8)
    ax.set_xlabel('predicted'); ax.set_ylabel('true'); ax.tick_params(length=0); [s.set_visible(False) for s in ax.spines.values()]
    ax.set_title(f'3 · Confusion matrix (accuracy {out["acc_5class"]*100:.1f}%)')

Yb = label_binarize(y, classes=range(5)); out['auc'] = {}
def p_roc(ax):
    for k in range(5):
        fpr, tpr, _ = roc_curve(Yb[:, k], proba[:, k]); a = auc(fpr, tpr); out['auc'][short[k]] = a
        ax.plot(fpr, tpr, color=PAL[k], lw=2, label=f'{short[k]}  AUC {a:.3f}')
    ax.plot([0, 1], [0, 1], color=GRID, ls='--', lw=1); ax.text(.55, .42, 'random guess', color=MUTE, fontsize=8, rotation=38)
    ax.set_xlabel('false positive rate'); ax.set_ylabel('true positive rate'); ax.set_xlim(0, 1); ax.set_ylim(0, 1.01)
    ax.set_title('4 · ROC curves (one class vs rest)'); ax.legend(frameon=False, fontsize=8, loc='lower right')

pr, rc, f1, _ = precision_recall_fscore_support(y, pred)
def p_prf(ax):
    xx = np.arange(5); wd = .26
    for i, (v, lab, c) in enumerate([(pr, 'Precision', '#1F2A22'), (rc, 'Recall', PAL[0]), (f1, 'F1', '#9FC9AE')]):
        ax.bar(xx + (i - 1) * wd, v * 100, wd - .03, color=c, label=lab)
    for k in range(5): ax.text(k, max(pr[k], rc[k], f1[k]) * 100 + 1.5, f'{f1[k]*100:.0f}', ha='center', fontsize=8.5, fontweight='bold')
    ax.set_xticks(xx); ax.set_xticklabels(short, fontsize=8.5); ax.set_ylim(50, 105); ax.set_ylabel('%')
    ax.legend(frameon=False, fontsize=8, ncol=3, loc='upper center', bbox_to_anchor=(.5, 1.0))
    ax.set_title('5 · Precision, recall, F1 per class (F1 labelled)')

Cs = np.logspace(-3, 3, 13)
tr, va = validation_curve(LR(), X, y, param_name='logisticregression__C', param_range=Cs, cv=cv)
bestC = Cs[va.mean(1).argmax()]; out['bestC'] = bestC; out['bestC_acc'] = va.mean(1).max()
def p_C(ax):
    ax.plot(Cs, tr.mean(1) * 100, color=MUTE, lw=2, marker='o', ms=4, label='training accuracy')
    ax.plot(Cs, va.mean(1) * 100, color=PAL[0], lw=2, marker='o', ms=4, label='validation accuracy (5-fold)')
    ax.fill_between(Cs, (va.mean(1) - va.std(1)) * 100, (va.mean(1) + va.std(1)) * 100, color=PAL[0], alpha=.15, lw=0)
    ax.axvline(bestC, color=MUTE, ls=':', lw=1); ax.text(bestC * 1.3, 52, f'best C ≈ {bestC:.1f}\n{va.mean(1).max()*100:.1f}%', fontsize=8.5, color=MUTE)
    ax.set_xscale('log'); ax.set_xlabel('C = 1/λ   (← stronger regularisation · weaker →)'); ax.set_ylabel('accuracy (%)'); ax.set_ylim(40, 100)
    ax.text(1.2e-3, 92, 'underfits', fontsize=8, color=MUTE); ax.text(900, 93.5, 'train–validation gap', fontsize=8, color=MUTE, ha='right')
    ax.set_title('6 · Regularisation strength vs accuracy'); ax.legend(frameon=False, fontsize=8, loc='lower right')

sizes, ltr, lva = learning_curve(LR(), X, y, cv=cv, train_sizes=np.linspace(.08, 1, 9), shuffle=True, random_state=0)
def p_learn(ax):
    ax.plot(sizes, ltr.mean(1) * 100, color=MUTE, lw=2, marker='o', ms=4, label='training accuracy')
    ax.plot(sizes, lva.mean(1) * 100, color=PAL[0], lw=2, marker='o', ms=4, label='validation accuracy')
    ax.fill_between(sizes, (lva.mean(1) - lva.std(1)) * 100, (lva.mean(1) + lva.std(1)) * 100, color=PAL[0], alpha=.15, lw=0)
    ax.set_xlabel('number of training leaves'); ax.set_ylabel('accuracy (%)'); ax.set_ylim(40, 101)
    ax.text(sizes[-1], lva.mean(1)[-1] * 100 - 6, 'still rising →\nmore data would help', fontsize=8, color=MUTE, ha='right')
    ax.set_title('7 · Learning curve'); ax.legend(frameon=False, fontsize=8, loc='lower right')

full = LR().fit(X, y); coef = full[-1].coef_          # standardised weights, shape 5 x 53
def p_coef(ax):
    c = coef[0]; idx = np.argsort(np.abs(c))[-12:]
    cols = [PAL[0] if v > 0 else PAL[1] for v in c[idx]]
    ax.barh(range(12), c[idx], color=cols, height=.65); ax.axvline(0, color=INK, lw=.8)
    ax.set_yticks(range(12)); ax.set_yticklabels([feat[i] for i in idx], fontsize=8); ax.tick_params(axis='y', length=0)
    ax.set_xlabel('weight (standardised features)')
    ax.text(.98, .04, 'green = pushes towards Healthy\norange = pushes away', transform=ax.transAxes, ha='right', fontsize=8, color=MUTE)
    ax.set_title('8 · Top 12 weights for the "Healthy" class')

conf = proba.max(1); ok = pred == y
def p_conf(ax):
    bins = np.linspace(.2, 1, 33)
    ax.hist(conf[ok], bins=bins, color=PAL[0], alpha=.8, label=f'correct ({ok.sum()})')
    ax.hist(conf[~ok], bins=bins, color=PAL[1], alpha=.85, label=f'wrong ({(~ok).sum()})')
    ax.set_xlabel('model confidence = highest predicted probability'); ax.set_ylabel('number of leaves')
    ax.set_title('9 · Confidence of right vs wrong predictions'); ax.legend(frameon=False, fontsize=8, loc='upper left')
    out['mean_conf_correct'] = conf[ok].mean(); out['mean_conf_wrong'] = conf[~ok].mean()

panels = [('lr_1_sigmoid', p_sigmoid), ('lr_2_decision_regions', p_regions), ('lr_3_confusion', p_cm), ('lr_4_roc', p_roc),
          ('lr_5_precision_recall_f1', p_prf), ('lr_6_regularisation_C', p_C), ('lr_7_learning_curve', p_learn),
          ('lr_8_weights', p_coef), ('lr_9_confidence', p_conf)]

**Graph 1 · Sigmoid.** Only one input: the brown-area % of the leaf. The S-curve turns a weighted sum into a probability between 0 and 1. Where it crosses 0.5 is the decision boundary. `class_weight='balanced'` gives both classes equal weight; without it the model just calls every leaf diseased, since 80% are.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.2)); panels[0][1](ax); fig.tight_layout()
fig.savefig('fig/lr_1_sigmoid.png', dpi=180, facecolor='white'); plt.show()

**Graph 2 · Decision regions.** Two features only (mean Blue, GLI). Each coloured area is where the model predicts that class. Every boundary is a straight line, which is why logistic regression is called a *linear* classifier. Two features overlap too much, so accuracy is only about 50%.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.2)); panels[1][1](ax); fig.tight_layout()
fig.savefig('fig/lr_2_decision_regions.png', dpi=180, facecolor='white'); plt.show()

**Graph 3 · Confusion matrix.** Rows are the true class, columns are the prediction, and the diagonal is correct. Healthy and YLCV are easy. The three brown-spot diseases get confused with each other.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.2)); panels[2][1](ax); fig.tight_layout()
fig.savefig('fig/lr_3_confusion.png', dpi=180, facecolor='white'); plt.show()

**Graph 4 · ROC curves.** For each class vs the rest: as the threshold drops, true positives rise (up) but so do false alarms (right). A curve hugging the top-left is good. AUC = area under the curve (1.0 perfect, 0.5 random).

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.2)); panels[3][1](ax); fig.tight_layout()
fig.savefig('fig/lr_4_roc.png', dpi=180, facecolor='white'); plt.show()

**Graph 5 · Precision / recall / F1.** Precision: when it says *Septoria*, how often is it right? Recall: of all real Septoria leaves, how many did it find? F1 is their balance.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.2)); panels[4][1](ax); fig.tight_layout()
fig.savefig('fig/lr_5_precision_recall_f1.png', dpi=180, facecolor='white'); plt.show()

**Graph 6 · Regularisation.** `C = 1/λ`. Small C = strong penalty on weights, so the model is too simple and underfits (both lines low). Large C = weak penalty, and training accuracy pulls away from validation (overfitting starts). Pick C where the green line peaks.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.2)); panels[5][1](ax); fig.tight_layout()
fig.savefig('fig/lr_6_regularisation_C.png', dpi=180, facecolor='white'); plt.show()

**Graph 7 · Learning curve.** The validation line is still rising at 1,200 leaves, so more data should keep improving the model. That's an argument for training on the full 87,848-image set.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.2)); panels[6][1](ax); fig.tight_layout()
fig.savefig('fig/lr_7_learning_curve.png', dpi=180, facecolor='white'); plt.show()

**Graph 8 · Weights.** Features are standardised, so weights can be compared. Positive (green) pushes a leaf towards *Healthy*, negative (orange) pushes away. Texture (Dissimilarity, Homogeneity) and certain colour bins matter most.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.2)); panels[7][1](ax); fig.tight_layout()
fig.savefig('fig/lr_8_weights.png', dpi=180, facecolor='white'); plt.show()

**Graph 9 · Confidence.** Logistic regression outputs probabilities, not just labels. Correct predictions are usually confident (near 1.0). Wrong ones cluster around 0.4–0.7, so low confidence is a useful warning sign.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.2)); panels[8][1](ax); fig.tight_layout()
fig.savefig('fig/lr_9_confidence.png', dpi=180, facecolor='white'); plt.show()

## 5 · All nine graphs on one sheet, and the summary numbers
The sheet is saved as `fig/LR_all_graphs.png`. The last cell downloads every graph as a zip.

In [ ]:
fig, axs = plt.subplots(3, 3, figsize=(19, 16.5))
for ax, (name, fn) in zip(axs.ravel(), panels): fn(ax)
fig.suptitle('Logistic Regression on 1,500 real PlantVillage tomato leaves  ·  53 features  ·  5-fold stratified CV', x=.01, ha='left', fontsize=16, fontweight='bold', y=.995)
fig.tight_layout(rect=(0, 0, 1, .975), h_pad=3, w_pad=2.5); fig.savefig('fig/LR_all_graphs.png', dpi=130, facecolor='white'); plt.show()
print('=== extra numbers ===')
print('AUC per class:', {k: round(v, 3) for k, v in out['auc'].items()})
print(f'best C = {bestC:.1f}  (CV accuracy {out["bestC_acc"]*100:.2f}%)')
print(f'learning curve: {sizes[0]} leaves → {lva.mean(1)[0]*100:.1f}%,  {sizes[-1]} leaves → {lva.mean(1)[-1]*100:.1f}%')
print(f'2 features (B mean, GLI) 5-class CV accuracy: {out["acc_2feat_cv"]*100:.1f}%')
print(f'mean confidence: correct {out["mean_conf_correct"]:.2f}, wrong {out["mean_conf_wrong"]:.2f}')
print('top Healthy weights:', [(feat[i], round(coef[0, i], 2)) for i in np.argsort(np.abs(coef[0]))[::-1][:6]])
json.dump({k: (v if not isinstance(v, np.floating) else float(v)) for k, v in out.items()}, open('lr_results.json', 'w'), indent=1, default=float)


In [ ]:
# download all graphs (Colab only)
try:
    from google.colab import files
    !zip -qj LR_graphs.zip fig/*.png
    files.download('LR_graphs.zip')
except ImportError:
    print('Not in Colab: graphs are in the fig/ folder')